In [1]:
# !pip install yfinance --upgrade -q

import yfinance as yf
import pandas as pd

def check_dca_signal(ticker_symbol, base_amount=1000):
    ticker = yf.Ticker(ticker_symbol)

    # 1. 获取价格与均线数据 (判断冷热情绪)
    hist = ticker.history(period="1y")
    current_price = hist['Close'].iloc[-1]
    ma_200 = hist['Close'].rolling(window=200).mean().iloc[-1]
    high_52w = hist['Close'].max()

    dist_from_ma200 = (current_price - ma_200) / ma_200
    drawdown = (current_price - high_52w) / high_52w

    # 2. 获取华尔街 EPS 趋势数据 (判断真实基本面)
    # eps_trend 包含: current, 7daysAgo, 30daysAgo, 60daysAgo, 90daysAgo
    # 行索引通常为: '0q' (本季), '+1q' (下季), '0y' (本年), '+1y' (明年)
    eps_trend = ticker.eps_trend
    eps_rev = ticker.eps_revisions

    # 提取明年('+1y')的 EPS 预测变化率 (代表长期基本面)
    eps_now = eps_trend.loc['+1y', 'current']
    eps_90d_ago = eps_trend.loc['+1y', '90daysAgo']
    eps_change_90d = (eps_now - eps_90d_ago) / abs(eps_90d_ago)

    # 提取近30天分析师上调/下调家数
    up_30d = eps_rev.loc['+1y', 'upLast30days']
    down_30d = eps_rev.loc['+1y', 'downLast30days']

    # 3. 决策树逻辑 (可根据你的风险偏好微调阈值)
    if drawdown <= -0.10 and eps_change_90d >= 0 and up_30d >= down_30d:
        zone = "🟢 黄金击球区 (错杀)"
        multiplier = 1.5
    elif drawdown <= -0.10 and (eps_change_90d <= -0.05 or down_30d > up_30d * 2):
        zone = "🔴 价值陷阱区 (基本面恶化)"
        multiplier = 0.0
    elif dist_from_ma200 >= 0.25 and eps_change_90d < 0.02:
        zone = "🟡 极度过热区 (纯拔估值)"
        multiplier = 0.5
    else:
        zone = "⚪ 舒适稳健区 (正常定投)"
        multiplier = 1.0

    invest_amount = base_amount * multiplier
    save_amount = max(0, base_amount - invest_amount)

    print(f"=== {ticker_symbol} 动态 DCA 诊断报告 ===")
    print(f"当前股价: ${current_price:.2f} | 偏离200日均线: {dist_from_ma200:.1%} | 高点回撤: {drawdown:.1%}")
    print(f"明年EPS预测: 现值 ${eps_now:.2f} vs 90天前 ${eps_90d_ago:.2f} (变动: {eps_change_90d:+.1%})")
    print(f"近30天修正家数: 上调 {int(up_30d)} 家 / 下调 {int(down_30d)} 家")
    print(f"当前状态: {zone}")
    print(f"建议操作: 本期买入 ${invest_amount:.0f} (系数 {multiplier}x)，存入蓄水池 ${save_amount:.0f}\n")

# 测试运行你的定投池
for stock in ['NVDA', 'MSFT', 'AAPL']:
    check_dca_signal(stock, base_amount=1000)

=== NVDA 动态 DCA 诊断报告 ===
当前股价: $221.30 | 偏离200日均线: 9.4% | 高点回撤: -7.5%
明年EPS预测: 现值 $15.91 vs 90天前 $12.76 (变动: +24.7%)
近30天修正家数: 上调 7 家 / 下调 0 家
当前状态: ⚪ 舒适稳健区 (正常定投)
建议操作: 本期买入 $1000 (系数 1.0x)，存入蓄水池 $0

=== MSFT 动态 DCA 诊断报告 ===
当前股价: $533.69 | 偏离200日均线: 23.1% | 高点回撤: -1.8%
明年EPS预测: 现值 $23.69 vs 90天前 $22.61 (变动: +4.8%)
近30天修正家数: 上调 2 家 / 下调 2 家
当前状态: ⚪ 舒适稳健区 (正常定投)
建议操作: 本期买入 $1000 (系数 1.0x)，存入蓄水池 $0

=== AAPL 动态 DCA 诊断报告 ===
当前股价: $342.96 | 偏离200日均线: 17.7% | 高点回撤: 0.0%
明年EPS预测: 现值 $9.58 vs 90天前 $9.69 (变动: -1.1%)
近30天修正家数: 上调 4 家 / 下调 3 家
当前状态: ⚪ 舒适稳健区 (正常定投)
建议操作: 本期买入 $1000 (系数 1.0x)，存入蓄水池 $0

